In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

RUN_MODELS = ["distilbert-base-uncased", "bert-base-uncased"]
RUN_TASKS = ["sst2", "cola", "mrpc"]
K_LIST = [10, 8, 6, 4, 3, 2]      # retained heads per layer (of 12)
OUT_DIR = "/kaggle/working/vo_results"
SMOKE = False                      # True: 1 checkpoint, K=[6,2] (sanity check only)
SEARCH = [
    "/kaggle/input/notebooks/*/*/ckpt/*/done.json",
    "/kaggle/input/*/ckpt/*/done.json",
    "/kaggle/input/**/done.json",
    "/kaggle/working/ckpt/**/done.json",
]
MAX_LEN = 128

import glob, json, time
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

if SMOKE:
    K_LIST, OUT_DIR = [6, 2], OUT_DIR + "_smoke"


def uniform_keep(score, k):
    """Keep the k highest-score heads in every layer. score: (L,H) -> bool (L,H)."""
    L, H = score.shape
    keep = np.zeros((L, H), bool)
    for l in range(L):
        keep[l, np.argsort(-score[l], kind="stable")[:k]] = True
    return keep


def vo_norm(Wo_h, Wv_h):
    """||W_V^h W_O^h||_2 from torch-layout slices: Wo_h (D, dh) = (W_O^h)^T, Wv_h (dh, D) = (W_V^h)^T.
    A = Wo_h @ Wv_h = (W_V^h W_O^h)^T.  Thin QR: Wo_h = Q1 R1, Wv_h^T = Q2 R2  ->  ||A||_2 = ||R1 R2^T||_2."""
    _, R1 = np.linalg.qr(Wo_h)
    _, R2 = np.linalg.qr(Wv_h.T)
    return float(np.linalg.norm(R1 @ R2.T, 2))


def mean_layer_tau(a, b):
    return float(np.mean([kendalltau(a[l], b[l])[0] for l in range(a.shape[0])]))


def kept_overlap(keep_a, keep_b):
    return float((keep_a & keep_b).sum() / max(keep_a.sum(), 1))


def selftest():
    rng = np.random.RandomState(0)
    # 1. QR shortcut == direct spectral norm of the product (real model shapes, rank-deficient, square)
    for (D, dh) in [(768, 64), (64, 16), (10, 10)]:
        for trial in range(6):
            Wo, Wv = rng.randn(D, dh), rng.randn(dh, D)
            if trial == 4: Wo[:, -1] = Wo[:, 0]                 # rank deficient W_O
            if trial == 5: Wv[0] = 0.0                          # zero row in W_V
            direct = np.linalg.norm(Wo @ Wv, 2)
            assert abs(vo_norm(Wo, Wv) - direct) <= 1e-8 * max(1.0, direct), (D, dh, trial)
    # 2. omega_h <= beta_V * beta_O   (the inequality used in the theory)
    for _ in range(20):
        Wo, Wv = rng.randn(96, 12), rng.randn(12, 96)
        assert vo_norm(Wo, Wv) <= np.linalg.norm(Wo, 2) * np.linalg.norm(Wv, 2) + 1e-9
    # 3. semantics: a head with a big W_O but a tiny W_V must rank BELOW a head with moderate both
    D, dh = 32, 4
    A_o, A_v = 5.0 * rng.randn(D, dh), 0.01 * rng.randn(dh, D)      # large op norm of W_O, negligible W_V
    B_o, B_v = 1.0 * rng.randn(D, dh), 1.0 * rng.randn(dh, D)
    assert np.linalg.norm(A_o, 2) > np.linalg.norm(B_o, 2)
    assert vo_norm(A_o, A_v) < vo_norm(B_o, B_v)
    # 4. keep masks: exactly k per layer, top scores, deterministic
    sc = rng.rand(5, 12)
    for k in (10, 6, 2):
        m = uniform_keep(sc, k)
        assert (m.sum(1) == k).all() and m.sum() == 5 * k
        for l in range(5):
            assert sc[l][m[l]].min() >= sc[l][~m[l]].max() if k < 12 else True
    assert (uniform_keep(sc, 4) == uniform_keep(sc, 4)).all()
    # 5. overlap / tau helpers
    a = uniform_keep(sc, 6)
    assert kept_overlap(a, a) == 1.0 and abs(mean_layer_tau(sc, sc) - 1.0) < 1e-12
    print("selftest OK (QR shortcut, bound, semantics, masks, helpers)")


selftest()


import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import matthews_corrcoef, f1_score

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEYS = {"sst2": ("sentence", None), "cola": ("sentence", None), "mrpc": ("sentence1", "sentence2")}


def get_attn(model):
    """list of (q, k, v, out) nn.Linear per layer; n_heads; hidden."""
    if hasattr(model, "distilbert"):
        ls = model.distilbert.transformer.layer
        acc = [(l.attention.q_lin, l.attention.k_lin, l.attention.v_lin, l.attention.out_lin) for l in ls]
        return acc, model.config.n_heads, model.config.dim
    ls = model.bert.encoder.layer
    acc = [(l.attention.self.query, l.attention.self.key, l.attention.self.value, l.attention.output.dense) for l in ls]
    return acc, model.config.num_attention_heads, model.config.hidden_size


def prep(task, tok):
    ds = load_dataset("nyu-mll/glue", task)
    k1, k2 = KEYS[task]

    def f(b):
        return tok(b[k1], b[k2], truncation=True, max_length=MAX_LEN) if k2 else tok(b[k1], truncation=True, max_length=MAX_LEN)

    split = ds["train"].train_test_split(test_size=0.05, seed=0)          # identical to fine-tuning script
    parts = {"train": split["train"], "val": ds["validation"]}
    cols = ["input_ids", "attention_mask", "label"]
    out = {}
    for n, d in parts.items():
        d = d.map(f, batched=True)
        out[n] = d.remove_columns([c for c in d.column_names if c not in cols])
    n_sub = min(5000, len(out["train"]))
    idx = np.random.RandomState(0).choice(len(out["train"]), n_sub, replace=False)
    out["train_sub"] = out["train"].select(sorted(idx.tolist()))
    return out


def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b["label"] for b in batch])
        enc = tok.pad([{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch], return_tensors="pt")
        enc["labels"] = labels
        return enc
    return collate


@torch.no_grad()
def predict(model, ds, collate, bs=128):
    model.eval()
    lg, lb = [], []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        lab = b.pop("labels")
        b = {k: v.to(dev_) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            lg.append(model(**b).logits.float().cpu())
        lb.append(lab)
    return torch.cat(lg).numpy(), torch.cat(lb).numpy()


def metrics(task, logits, labels):
    p = logits.argmax(1)
    loss = torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
    m = {"acc": float((p == labels).mean()), "loss": loss}
    m["mcc"] = float(matthews_corrcoef(labels, p)) if task == "cola" else np.nan
    m["f1"] = float(f1_score(labels, p)) if task == "mrpc" else np.nan
    return m


def head_scores(model):
    """frob / op / chain / kappa from the DENSE weights (verbatim from prune_eval.py)."""
    acc, H, D = get_attn(model)
    dh = D // H
    L = len(acc)
    S = {k: np.zeros((L, H)) for k in ["frob", "op", "chain", "kappa", "kappa_row"]}
    t0 = time.time()
    with torch.no_grad():
        for l, (q, k, v, o) in enumerate(acc):
            Wq, Wk, Wv, Wo = [m.weight.detach().float().cpu() for m in (q, k, v, o)]
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                Woh = Wo[:, sl]
                sv = torch.linalg.svdvals(Woh)
                S["op"][l, h] = sv[0].item()
                S["frob"][l, h] = torch.linalg.matrix_norm(Woh, "fro").item()
                S["kappa"][l, h] = (sv[0] ** 2 / (sv ** 2).mean()).item()
                rn = (Woh ** 2).sum(1)
                S["kappa_row"][l, h] = (rn.max() / (rn.mean() + 1e-12)).item()
                bq = torch.linalg.matrix_norm(Wq[sl, :], 2).item()
                bk = torch.linalg.matrix_norm(Wk[sl, :], 2).item()
                bv = torch.linalg.matrix_norm(Wv[sl, :], 2).item()
                S["chain"][l, h] = bq * bk * bv * sv[0].item()
    S["_time_svd_s"] = time.time() - t0
    return S


def vo_scores(model):
    """omega_h = ||W_V^h W_O^h||_2 for every head (exact, float64, thin QR). Returns ((L,H) array, seconds)."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    out = np.zeros((L, H))
    t0 = time.time()
    with torch.no_grad():
        for l, (_, _, v, o) in enumerate(acc):
            Wv = v.weight.detach().double().cpu().numpy()      # (H*dh, D): rows of head h = (W_V^h)^T
            Wo = o.weight.detach().double().cpu().numpy()      # (D, H*dh): columns of head h = (W_O^h)^T
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                out[l, h] = vo_norm(Wo[:, sl], Wv[sl, :])
    return out, time.time() - t0


class Pruner:
    """Applies a keep-mask by zeroing the out-projection columns of pruned heads (exactly removes the head)."""
    def __init__(self, model):
        self.model = model
        self.acc, self.H, self.D = get_attn(model)
        self.dh = self.D // self.H
        self.W0 = [o.weight.detach().clone() for (_, _, _, o) in self.acc]

    def apply(self, keep):
        with torch.no_grad():
            for l, (_, _, _, o) in enumerate(self.acc):
                o.weight.copy_(self.W0[l])
                for h in range(self.H):
                    if not keep[l, h]:
                        o.weight[:, h * self.dh:(h + 1) * self.dh] = 0.0


def discover():
    runs = {}
    for pat in SEARCH:
        for p in glob.glob(pat, recursive=True):
            try:
                r = json.load(open(p))
            except Exception:
                continue
            runs[(r["model"], r["task"], r["seed"])] = os.path.dirname(p)
    sel = {k: v for k, v in runs.items() if k[0] in RUN_MODELS and k[1] in RUN_TASKS}
    if not sel:
        print("No matching checkpoints. All done.json files seen:", sorted(runs.keys()))
    sel = dict(sorted(sel.items()))
    if SMOKE:
        sel = dict(list(sel.items())[:1])
    return sel


def run_checkpoint(key, path, cache):
    mname, task, seed = key
    tag = f"{mname}_{task}_seed{seed}"
    odir = f"{OUT_DIR}/{tag}"
    if os.path.exists(f"{odir}/DONE"):
        print("skip (done):", tag); return
    os.makedirs(odir, exist_ok=True)
    if (mname, task) not in cache:
        tok = AutoTokenizer.from_pretrained(path)
        cache[(mname, task)] = (tok, prep(task, tok), make_collate(tok))
    tok, data, collate = cache[(mname, task)]
    model = AutoModelForSequenceClassification.from_pretrained(path).to(dev_).eval()
    pr = Pruner(model)
    L, H = len(pr.acc), pr.H
    print(f"\n=== {tag}: L={L} H={H}")

    S = head_scores(model)
    VO, t_vo = vo_scores(model)
    tau_vo_op = mean_layer_tau(VO, S["op"])
    rho_vo_op = float(spearmanr(VO.ravel(), S["op"].ravel())[0])
    print(f"  vo scores in {t_vo:.2f}s | mean per-layer Kendall tau(vo, op) = {tau_vo_op:.3f} | Spearman over all heads = {rho_vo_op:.3f}")

    rows, preds = [], {}

    def evaluate(method, k, draw, keep, extra=None):
        pr.apply(keep)
        lv, yv = predict(model, data["val"], collate)
        lt, yt = predict(model, data["train_sub"], collate)
        mv, mt = metrics(task, lv, yv), metrics(task, lt, yt)
        row = dict(model=mname, task=task, seed=seed, method=method, k=k, draw=draw,
                   n_kept=int(keep.sum()), frac_kept=float(keep.sum()) / (L * H),
                   kept_per_layer=" ".join(str(int(x)) for x in keep.sum(1)),
                   val_acc=mv["acc"], val_loss=mv["loss"], val_mcc=mv["mcc"], val_f1=mv["f1"],
                   train_acc=mt["acc"], train_loss=mt["loss"],
                   gap_loss=mv["loss"] - mt["loss"], gap_acc=mt["acc"] - mv["acc"],
                   C_fro=float(np.sqrt((S["frob"][keep] ** 2).sum())), C_op=float(S["op"][keep].sum()),
                   score_time_svd_s=S["_time_svd_s"], score_time_vo_s=t_vo)
        if extra: row.update(extra)
        rows.append(row)
        preds[f"{method}|{k}|{draw}"] = (lv.argmax(1)).astype(np.int8)
        if "labels" not in preds: preds["labels"] = yv.astype(np.int8)

    evaluate("dense", H, 0, np.ones((L, H), bool))
    # sanity: the dense model evaluated here must reproduce the score stored by the fine-tuning script
    try:
        ref = json.load(open(f"{path}/done.json"))["val"]["acc"]
        dd = abs(rows[0]["val_acc"] - ref)
        print(f"  dense check vs done.json: |diff| = {dd:.4f}  " + ("OK" if dd < 0.003 else "!! WARNING: mismatch, wrong checkpoint or setup"))
    except Exception as e:
        print("  (dense check skipped:", type(e).__name__, ")")

    for k in K_LIST:
        keep_vo, keep_op = uniform_keep(VO, k), uniform_keep(S["op"], k)
        ov = kept_overlap(keep_vo, keep_op)
        evaluate("asicr_uniform", k, 0, keep_op)                                   # re-evaluated for a same-run comparison
        evaluate("vo_uniform", k, 0, keep_vo, {"overlap_vo_op": ov})               # omega_h = ||W_V W_O||_2
        print(f"  k={k}: kept-set overlap vo/op = {ov:.2f}")

    pd.DataFrame(rows).to_csv(f"{odir}/results.csv", index=False)
    np.savez_compressed(f"{odir}/val_preds.npz", **preds)
    np.savez_compressed(f"{odir}/vo_scores.npz", vo=VO, op=S["op"], frob=S["frob"], chain=S["chain"])
    json.dump(dict(tag=tag, mean_layer_tau_vo_op=tau_vo_op, spearman_vo_op=rho_vo_op, vo_seconds=t_vo, K_LIST=K_LIST),
              open(f"{odir}/meta.json", "w"), indent=2)
    open(f"{odir}/DONE", "w").write("ok")
    del model, pr
    torch.cuda.empty_cache()


def main():
    runs = discover()
    print(f"Found {len(runs)} checkpoints:")
    for k, v in runs.items():
        print("  ", k, "->", v)
    assert runs, "No checkpoints found: add the fine-tuning notebook outputs via Add Input -> Notebook Output Files"
    cache = {}
    for key, path in runs.items():
        run_checkpoint(key, path, cache)
    allr = [pd.read_csv(p) for p in glob.glob(f"{OUT_DIR}/*/results.csv")]
    if allr:
        A = pd.concat(allr)
        A.to_csv(f"{OUT_DIR}/ALL_vo_results.csv", index=False)
        A["primary"] = np.where(A.task == "cola", A.val_mcc, np.where(A.task == "mrpc", A.val_f1, A.val_acc)) * 100
        s = A[A.method != "dense"].groupby(["model", "task", "method", "k"])["primary"].mean().unstack("k")
        print("\nMean score x100 (vo_uniform vs asicr_uniform):")
        print(s.round(1).to_string())
    print("\nALL DONE ->", OUT_DIR)


if __name__ == "__main__" or True:
    main()

selftest OK (QR shortcut, bound, semantics, masks, helpers)
Found 18 checkpoints:
   ('bert-base-uncased', 'sst2', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed0
   ('bert-base-uncased', 'sst2', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed1
   ('bert-base-uncased', 'sst2', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed2
   ('distilbert-base-uncased', 'cola', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed0
   ('distilbert-base-uncased', 'cola', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed1
   ('distilbert-base-uncased', 'cola', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed2
   ('distilbert-base-uncased', 'cola', 3) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (63 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_sst2_seed3: L=6 H=12
  vo scores in 0.50s | mean per-layer Kendall tau(vo, op) = 0.525 | Spearman over all heads = 0.559
  dense check vs done.json: |diff| = 0.0000  OK
  k=10: kept-set overlap vo/op = 0.97
  k=8: kept-set overlap vo/op = 0.85
  k=6: kept-set overlap vo/op = 0.67
  k=4: kept-set overlap vo/op = 0.58
  k=3: kept-set overlap vo/op = 0.50
  k=2: kept-set overlap vo/op = 0.50


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_sst2_seed4: L=6 H=12
  vo scores in 0.49s | mean per-layer Kendall tau(vo, op) = 0.520 | Spearman over all heads = 0.554
  dense check vs done.json: |diff| = 0.0000  OK
  k=10: kept-set overlap vo/op = 0.97
  k=8: kept-set overlap vo/op = 0.85
  k=6: kept-set overlap vo/op = 0.67
  k=4: kept-set overlap vo/op = 0.58
  k=3: kept-set overlap vo/op = 0.50
  k=2: kept-set overlap vo/op = 0.50

Mean score x100 (vo_uniform vs asicr_uniform):
k                                             2     3     4     6     8     10
model                   task method                                           
bert-base-uncased       sst2 asicr_uniform  66.4  73.2  79.0  88.7  91.4  91.4
                             vo_uniform     77.3  78.8  79.5  84.9  87.4  90.4
distilbert-base-uncased cola asicr_uniform   7.9   7.9  11.6  25.0  33.2  41.6
                             vo_uniform      2.4   7.3   7.2  14.8  32.6  43.8
                        mrpc asicr_uniform   0.0   0.0   